In [ ]:
def main(datasources, start_date, end_date):
    """端到端单频率 Transformer 优化版 v3。

    v3 核心改进 (基于 v2):
    1. 因果注意力掩码: 修复数据泄露 — 第 t 个 bar 只能看到 t 及之前的信息 (v2 最严重 bug)
    2. 最后时刻池化: 因果掩码下用 h[:, -1, :] 替代注意力池化, 更符合时序逻辑
    3. 标签鲁棒性: 过滤异常收益 (|r|>0.5) 和无效 close, 避免停牌填充失真
    4. 截面标准化边界: 样本不足 (<5) 的日期用全局统计量兜底, 不跳过
    5. 验证集扩展: 6 个月 (2023-07~2023-12), 覆盖更多市场状态
    6. IC 平滑早停: 3 日滚动 IC 均值, 避免单日波动频繁触发
    7. 增强诊断: 输出 IC 标准差、IR (IC均值/标准差)
    8. 异常处理: 空查询、数据不足等情况更稳健
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    from copy import deepcopy
    import structlog

    logger = structlog.get_logger()

    # ---------- 配置 ----------
    TRAIN_TABLE = "bigalpha_2026_stock_bar30m"
    INFER_TABLE = datasources["bar30m"]
    TRAIN_START, TRAIN_END = "2020-01-01", "2023-06-30"         # 训练集: ~3.5 年
    VAL_START, VAL_END = "2023-07-01", "2023-12-31"             # 验证集: 6 个月
    SEQ_LEN = 64
    EPOCHS, BATCH, LR, SEED = 16, 512, 2e-3, 42
    MAX_TRAIN_INSTRUMENTS = 500
    PATIENCE = 5
    MIN_VAL_SAMPLES = 5000
    IC_SMOOTH_WINDOW = 3                                         # IC 平滑窗口

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info("运行设备", device=str(device), train_table=TRAIN_TABLE, infer_table=INFER_TABLE)

    # ---------- 动态探测可用字段 ----------
    try:
        _probe = dai.query(f"SELECT * FROM {TRAIN_TABLE} LIMIT 1").df()
        _available = set(_probe.columns)
    except Exception:
        _available = {"open", "high", "low", "close", "volume", "amount",
                      "bid_price1", "ask_price1", "bid_volume1", "ask_volume1"}
    ALL_PRICE = ["open", "high", "low", "close", "pre_close",
                 "bid_price1", "bid_price2", "bid_price3",
                 "ask_price1", "ask_price2", "ask_price3"]
    ALL_VOL = ["volume", "amount",
               "bid_volume1", "bid_volume2", "bid_volume3",
               "ask_volume1", "ask_volume2", "ask_volume3"]
    PRICE_COLS = [c for c in ALL_PRICE if c in _available]
    VOL_COLS = [c for c in ALL_VOL if c in _available]
    FEATURE_COLS = PRICE_COLS + VOL_COLS
    N_FEAT = len(FEATURE_COLS)
    logger.info("使用特征", n_feat=N_FEAT, features=FEATURE_COLS)
    assert N_FEAT >= 2, "可用特征不足"

    # ---------- 模型 (因果注意力 + 最后时刻池化) ----------
    class StockTransformer(nn.Module):
        def __init__(self, n_feat, d_model=128, nhead=4, nlayers=3, dim_ff=256,
                     dropout=0.2, seq_len=SEQ_LEN):
            super().__init__()
            self.proj = nn.Linear(n_feat, d_model)
            # 固定 sin/cos 位置编码 (节省可学习参数, 更好泛化)
            pos = torch.zeros(1, seq_len, d_model)
            position = torch.arange(seq_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() *
                            (-np.log(10000.0) / d_model))
            pos[0, :, 0::2] = torch.sin(position * div)
            pos[0, :, 1::2] = torch.cos(position * div)
            self.register_buffer("pos", pos)
            # 因果掩码: 上三角为 -inf, 确保第 t 个位置只能看到 0..t
            # 修复 v2 双向注意力的数据泄露问题
            mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
            self.register_buffer("causal_mask", mask)
            layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout,
                                               batch_first=True, activation="gelu")
            self.encoder = nn.TransformerEncoder(layer, nlayers)
            # 最后时刻池化: 因果掩码下, 最后一个位置已看到所有历史信息
            # 比 v2 的注意力池化更简单、更符合时序逻辑
            self.head = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Dropout(dropout),
                nn.Linear(d_model, 1)
            )

        def forward(self, x):                                       # (B, L, N_FEAT) -> (B,)
            h = self.encoder(self.proj(x) + self.pos,
                             mask=self.causal_mask)                  # (B, L, D) 因果注意力
            h_last = h[:, -1, :]                                    # (B, D) 最后时刻表征
            return self.head(h_last).squeeze(-1)                    # (B,)

    # ---------- 数据: 切窗口 + NaN处理 + 截面标准化 ----------
    def build_dataset(table, instruments, sd, ed, mode):
        """构建数据集并做截面标准化。

        mode: 'train', 'val', 'infer'
        返回:
          - mode='train': X, y_raw, y_std, keys_df
          - mode='val':   X, y_raw, keys_df
          - mode='infer': X, keys_df
        """
        t0 = time.time()
        buf = (pd.to_datetime(sd) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")
        sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {table} ORDER BY instrument, date"
        try:
            df = dai.query(sql, filters={"date": [buf, ed], "instrument": instruments}).df()
        except Exception as e:
            logger.error("SQL查询失败", error=str(e), table=table)
            if mode == "infer":
                return np.empty((0, SEQ_LEN, N_FEAT), np.float32), \
                       pd.DataFrame(columns=["date", "instrument"])
            raise

        if df.empty:
            logger.warning("查询结果为空", table=table, sd=sd, ed=ed)
            if mode == "infer":
                return np.empty((0, SEQ_LEN, N_FEAT), np.float32), \
                       pd.DataFrame(columns=["date", "instrument"])
            raise RuntimeError(f"build_dataset 查询结果为空 (mode={mode}, {sd}~{ed})")

        # NaN 处理: 价格 ffill + 填0; 量 log1p(clip)
        for c in PRICE_COLS:
            if c in df.columns:
                df[c] = df.groupby("instrument")[c].ffill()
                df[c] = df[c].fillna(0)
        for c in VOL_COLS:
            if c in df.columns:
                df[c] = np.log1p(df[c].clip(lower=0))

        sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
        wins, ys_raw, keys = [], [], []
        for ins, sub in df.groupby("instrument", sort=False):
            if len(sub) <= SEQ_LEN:
                continue
            feats = sub[FEATURE_COLS].to_numpy(np.float32)
            day = sub["date"].dt.normalize().to_numpy()
            close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
            close_px = sub["close"].to_numpy(np.float64)[close_pos]
            dates = day[close_pos]
            for k, p in enumerate(close_pos):
                d = pd.Timestamp(dates[k])
                if p + 1 < SEQ_LEN or d < sd_ts or d > ed_ts:
                    continue
                label = None
                # 标签鲁棒性: 检查下一交易日 close 是否有效, 过滤异常收益
                if k + 1 < len(close_pos) and close_px[k] > 0 and close_px[k + 1] > 0:
                    r = close_px[k + 1] / close_px[k] - 1.0        # T+1 收益
                    # 过滤: NaN/inf + 极端值 (|r|>0.5, 多为停牌填充或数据错误)
                    if np.isfinite(r) and abs(r) < 0.5:
                        label = np.float32(r)
                if mode in ("train", "val") and label is None:
                    continue
                wins.append(feats[p - SEQ_LEN + 1: p + 1])
                ys_raw.append(label if label is not None else np.float32(0.0))
                keys.append((d, ins))

        if not keys:
            if mode == "infer":
                logger.warning("推理集无样本", sd=sd, ed=ed)
                return np.empty((0, SEQ_LEN, N_FEAT), np.float32), \
                       pd.DataFrame(columns=["date", "instrument"])
            raise RuntimeError(f"build_dataset 无样本 (mode={mode}, {sd}~{ed})")

        X = np.stack(wins).astype(np.float32)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        keys_df = pd.DataFrame(keys, columns=["date", "instrument"])

        # ---------- 截面标准化 ----------
        # 预计算全局统计量 (用于样本不足的日期兜底, v2 直接跳过会导致分布不一致)
        global_mean = X.mean(axis=0, keepdims=True)
        global_std = X.std(axis=0, keepdims=True) + 1e-6

        for d in keys_df["date"].unique():
            mask = (keys_df["date"] == d).to_numpy()
            n_day = mask.sum()
            X_d = X[mask]
            if n_day < 5:
                # 样本不足: 使用全局统计量兜底, 保证输入分布一致
                X[mask] = ((X_d - global_mean) / global_std).astype(np.float32)
            else:
                mean_d = X_d.mean(axis=0, keepdims=True)
                std_d = X_d.std(axis=0, keepdims=True) + 1e-6
                X[mask] = ((X_d - mean_d) / std_d).astype(np.float32)

        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        t1 = round(time.time() - t0, 2)
        logger.info(f"{mode} 集构建完成", samples=len(keys), elapsed=t1)

        if mode == "train":
            # 标签截面标准化 (使 MSE 对齐 Rank IC)
            y_raw = np.array(ys_raw, np.float32)
            y_std = y_raw.copy()
            global_y_mean = y_raw.mean()
            global_y_std = y_raw.std() + 1e-6
            for d in keys_df["date"].unique():
                mask = (keys_df["date"] == d).to_numpy()
                n_day = mask.sum()
                if n_day < 5:
                    # 样本不足: 使用全局统计量兜底
                    y_std[mask] = ((y_raw[mask] - global_y_mean) /
                                   global_y_std).astype(np.float32)
                else:
                    yd = y_std[mask]
                    y_std[mask] = ((yd - yd.mean()) / (yd.std() + 1e-6)).astype(np.float32)
            y_std = np.nan_to_num(y_std, nan=0.0)
            return X, y_raw, y_std, keys_df
        elif mode == "val":
            return X, np.array(ys_raw, np.float32), keys_df
        else:  # infer
            return X, keys_df

    def pool(table, sd, ed):
        try:
            df = dai.query(f"SELECT DISTINCT instrument FROM {table}",
                           filters={"date": [sd, ed]}).df()
            return df["instrument"].tolist()
        except Exception as e:
            logger.error("获取标的列表失败", error=str(e), table=table)
            return []

    # ---------- 获取标的列表 ----------
    all_inst = pool(TRAIN_TABLE, TRAIN_START, TRAIN_END)[:MAX_TRAIN_INSTRUMENTS]
    if not all_inst:
        raise RuntimeError("无法获取训练标的列表")
    logger.info("训练标的数", n=len(all_inst))

    # ---------- 构建训练集 ----------
    logger.info("构建训练集", start=TRAIN_START, end=TRAIN_END)
    Xtr, ytr_raw, ytr_std, tr_keys = build_dataset(
        TRAIN_TABLE, all_inst, TRAIN_START, TRAIN_END, "train")
    lo, hi = np.percentile(ytr_std, [1, 99])
    ytr_std = np.clip(ytr_std, lo, hi)
    logger.info("训练集", samples=len(Xtr), days=tr_keys["date"].nunique())

    # ---------- 构建验证集 (6 个月) ----------
    logger.info("构建验证集", start=VAL_START, end=VAL_END)
    Xval, yval_raw, val_keys = None, None, None
    try:
        Xval, yval_raw, val_keys = build_dataset(
            TRAIN_TABLE, all_inst, VAL_START, VAL_END, "val")
        logger.info("验证集", samples=len(Xval), days=val_keys["date"].nunique())
        if len(Xval) < MIN_VAL_SAMPLES:
            logger.warning(f"验证集样本不足 ({len(Xval)} < {MIN_VAL_SAMPLES}), 放弃早停")
            Xval, yval_raw, val_keys = None, None, None
    except RuntimeError as e:
        logger.warning("验证集构建失败, 放弃早停", error=str(e))

    use_validation = Xval is not None

    # ---------- 模型 ----------
    model = StockTransformer(N_FEAT).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    logger.info("可训练参数量", n_params=n_params)
    assert 100_000 <= n_params <= 100_000_000, f"参数量 {n_params} 不在 [10万, 1亿] 范围内"

    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr_std)),
                        batch_size=BATCH, shuffle=True, pin_memory=(device.type == "cuda"))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS,
        steps_per_epoch=len(loader), pct_start=0.1)
    loss_fn = nn.MSELoss()

    # ---------- 辅助: 计算每日 Rank IC + IR ----------
    def calc_daily_ic(keys_df, preds, raw_y):
        """计算每日 Spearman Rank IC, 返回均值、标准差、IR、逐日列表。"""
        df = keys_df.copy()
        df["pred"] = preds
        df["y"] = raw_y
        daily_ics = []
        for _, g in df.groupby("date"):
            if len(g) > 10:
                ic = g["pred"].corr(g["y"], method="spearman")
                if np.isfinite(ic):
                    daily_ics.append(ic)
        mean_ic = float(np.mean(daily_ics)) if daily_ics else 0.0
        std_ic = float(np.std(daily_ics)) if daily_ics else 0.0
        ir = mean_ic / (std_ic + 1e-8) if std_ic > 1e-8 else 0.0
        return mean_ic, std_ic, ir, daily_ics

    # ---------- 训练 (带 IC 平滑早停) ----------
    best_val_ic_smooth, best_state, best_epoch = -999, None, 0
    patience_counter = 0
    val_ic_history = []                                              # 用于 IC 平滑

    model.train()
    for ep in range(EPOCHS):
        t_ep, tot_loss, nb = time.time(), 0.0, 0
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            tot_loss += loss.item()
            nb += 1

        avg_loss = round(tot_loss / max(nb, 1), 8)
        lr_now = round(opt.param_groups[0]["lr"], 6)

        # 训练集 IC + IR
        model.eval()
        with torch.no_grad():
            tr_preds = []
            for i in range(0, len(Xtr), BATCH):
                xb = torch.from_numpy(Xtr[i:i + BATCH]).to(device)
                tr_preds.append(model(xb).cpu().numpy())
            tr_preds = np.concatenate(tr_preds)
        mean_tr_ic, std_tr_ic, tr_ir, _ = calc_daily_ic(tr_keys, tr_preds, ytr_raw)
        model.train()

        # 验证集 IC + IR (如果有)
        val_ic_str = ""
        if use_validation:
            model.eval()
            with torch.no_grad():
                val_preds = []
                for i in range(0, len(Xval), BATCH):
                    xb = torch.from_numpy(Xval[i:i + BATCH]).to(device)
                    val_preds.append(model(xb).cpu().numpy())
                val_preds = np.concatenate(val_preds)
            mean_val_ic, std_val_ic, val_ir, _ = calc_daily_ic(
                val_keys, val_preds, yval_raw)
            val_ic_str = (f", val_ic={mean_val_ic:.4f}, "
                          f"val_ir={val_ir:.4f}")
            model.train()

            # IC 平滑早停: 使用最近 N 日 IC 均值, 避免单日波动频繁触发
            val_ic_history.append(mean_val_ic)
            if len(val_ic_history) >= IC_SMOOTH_WINDOW:
                smooth_ic = float(np.mean(val_ic_history[-IC_SMOOTH_WINDOW:]))
            else:
                smooth_ic = float(np.mean(val_ic_history))

            if smooth_ic > best_val_ic_smooth + 0.001:
                best_val_ic_smooth = smooth_ic
                best_state = deepcopy(model.state_dict())
                best_epoch = ep + 1
                patience_counter = 0
            else:
                patience_counter += 1

        logger.info(f"epoch {ep + 1}/{EPOCHS}",
                    loss=avg_loss, tr_ic=round(mean_tr_ic, 4),
                    tr_ir=round(tr_ir, 4),
                    extra=val_ic_str, lr=lr_now,
                    elapsed=round(time.time() - t_ep, 2))

        if use_validation and patience_counter >= PATIENCE:
            logger.info("早停触发", best_epoch=best_epoch,
                        best_val_ic_smooth=round(best_val_ic_smooth, 4))
            model.load_state_dict(best_state)
            break

    if not use_validation or best_state is None:
        logger.info("无早停, 使用最终模型")

    # ---------- 训练集 IC 诊断 (增强: IC std, IR) ----------
    model.eval()
    with torch.no_grad():
        tr_preds = []
        for i in range(0, len(Xtr), BATCH):
            xb = torch.from_numpy(Xtr[i:i + BATCH]).to(device)
            tr_preds.append(model(xb).cpu().numpy())
        tr_preds = np.concatenate(tr_preds)
    final_tr_ic, final_tr_std, final_tr_ir, tr_daily_ics = calc_daily_ic(
        tr_keys, tr_preds, ytr_raw)
    logger.info("训练集最终 IC",
                mean_rank_ic=round(final_tr_ic, 4),
                std_ic=round(final_tr_std, 4),
                ir=round(final_tr_ir, 4),
                n_days=len(tr_daily_ics))
    if final_tr_ic < 0:
        logger.warning("训练集 IC 为负! 模型未学到有效信号")

    # ---------- 推理 ----------
    logger.info("构建测试集并预测", table=INFER_TABLE,
                start=str(start_date), end=str(end_date))
    infer_inst = pool(INFER_TABLE, start_date, end_date)
    if not infer_inst:
        raise RuntimeError("推理集无可用标的")
    Xte, idx_df = build_dataset(
        INFER_TABLE, infer_inst, start_date, end_date, "infer")
    if len(idx_df) == 0:
        raise RuntimeError("推理集构建无样本")
    model.eval()
    preds = []
    Xte_t = torch.from_numpy(Xte)
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xb = Xte_t[i:i + BATCH].to(device)
            preds.append(model(xb).cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    # ---------- 对齐中证1000 + 规范输出 ----------
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasources = {
        "bar30m": "bigalpha_2026_stock_bar30m",
    }

    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        show=True,
    )
